# Calculating J

In [2]:
import torch

# The following is vibe-coded with Gemini; fixed by Kevin Lim and Zihan Guo

# Use float64 for maximum precision with exponential terms
torch.set_default_dtype(torch.float64)

def calculate_J(R1, R2, S0, theta):

    # Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    nus1 = torch.cat([nus1, torch.tensor([S0])])
    nus2 = torch.cat([torch.tensor([S0]), nus2])


    # Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)

    # Left Side
    c_v1_unit = []
    dist0 = nus1[1] - nus1[0]
    c_v1_unit.append((-torch.exp(-dist0/sigs1[0]), torch.exp(dist0/sigs1[0])))

    # recursively calculating c^1
    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+2] - nus1[j+1]
        c_v1_unit.append((0.5 * (P - sigs1[j+1]*S) * torch.exp(-dist/sigs1[j+1]),
                          0.5 * (P + sigs1[j+1]*S) * torch.exp(dist/sigs1[j+1])))

    # Right Side
    c_v2_unit = [None] * R2
    dist_u = nus2[-1] - nus2[-2]
    c_v2_unit[-1] = ((torch.exp(dist_u/sigs2[-1]), -torch.exp(-dist_u/sigs2[-1])))

    # recursively calculating c^2
    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j] - nus2[j-1]
        c_v2_unit[j-1] = (0.5 * (P - sigs2[j-1]*S) * torch.exp(dist/sigs2[j-1]),
                          0.5 * (P + sigs2[j-1]*S) * torch.exp(-dist/sigs2[j-1]))

    # Unscaled left-side value at S0
    v1 = c_v1_unit[-1][0] + c_v1_unit[-1][1]

    # Unscaled right-side value at S0
    v2 = c_v2_unit[0][0] + c_v2_unit[0][1]

    # Unscaled left-side derivative at S0
    Dk_v1 = (1/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])

    # Unscaled right-side derivative at S0
    Dk_v2 = (1/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])

    # Values for lambda1 and lambda2
    lam1 = v2 / (Dk_v1 * v2 - v1 * Dk_v2)
    lam2 = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    jumps = []

    # Left Side Jumps
    for j in range(R1 - 1):
        v_left = (1/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+2] - nus1[j+1]
        v_right = (1/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                             c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_right - v_left)

    # Jump at S0
    v1_S0 = (1/sigs1[-1]**2) * (c_v1[-1][0] + c_v1[-1][1])
    v2_S0 = (1/sigs2[0]**2) * (c_v2[0][0] + c_v2[0][1])

    jumps.append(v2_S0 - v1_S0)

    # Right Side Jumps
    for j in range(R2 - 1):
        v_right = (1/sigs2[j+1]**2) * (c_v2[j+1][0] + c_v2[j+1][1])
        dist = nus2[j+1] - nus2[j]
        v_left = (1/sigs2[j]**2) * (c_v2[j][0]*torch.exp(-dist/sigs2[j]) +
                                             c_v2[j][1]*torch.exp(dist/sigs2[j]))
        jumps.append(v_right - v_left)

    j = torch.sum(torch.stack(jumps) ** 2)

    C_K = []

    for i in range(R1):
        C_K.append(c_v1[i][0] + c_v1[i][1])

    C_K.append(c_v1[-1][0] * torch.exp((nus1[-1] - nus1[-2]) / sigs1[-1]) + c_v1[-1][1] * torch.exp(-(nus1[-1] - nus1[-2]) / sigs1[-1]))

    for i in range(R2):
        C_K.append(c_v2[i][0] + c_v2[i][1])

    C_K = torch.stack(C_K)

    return j, lam1, lam2, C_K

# Built-in Line Search with torch.optim.LBFGS (constrained)

**Log barrier method**

$\min_{\theta} L(\theta) = J(\theta) + \mu \sum_{i=1}^{R_1 + R_2 + 1} \left[ -\ln\left(C(K_i) - \text{Bid}_i\right) - \ln\left(\text{Ask}_i - C(K_i)\right) \right]$

Minimize $f(x)$ s.t. $g_i(x) \le 0, \forall i = 1, \dots, m$

Modified objective function: $f(x) - \mu \sum_{i=1}^m \log(-g_i(x))$

In [3]:
def obtain_nus(R1, R2, S0, theta):
    nus1 = theta[0 : R1]
    nus2 = theta[2 * R1 : 2 * R1 + R2]
    return nus1, nus2

def obtain_sigs(R1, R2, theta):
    sigs1 = theta[R1 : 2 * R1]
    sigs2 = theta[2 * R1 + R2 : 2 * R1 + 2 * R2]
    return sigs1, sigs2

def C_K(R1, R2, S0, theta):
    _, _, _, C_K = calculate_J(R1, R2, S0, theta)
    return C_K

# nus do not include S0
def obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2):
    theta = torch.cat([nus1, sigs1, nus2, sigs2])
    return theta

In [4]:
def generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector):

    nus1, nus2 = obtain_nus(R1, R2, S0, theta)

    S0_tensor = torch.tensor([S0], dtype=nus1.dtype, device=nus1.device)
    strikes = torch.cat([nus1, S0_tensor, nus2])

    # (N x 3) matrix
    bid_ask_matrix = torch.zeros((R1 + R2 + 1, 3), dtype=theta.dtype, device=theta.device)

    bid_ask_matrix[:, 0] = strikes
    model_prices = C_K(R1, R2, S0, theta)
    bid_ask_matrix[:, 1] = torch.clamp(model_prices - eps_bid_vector, min=0.0)
    bid_ask_matrix[:, 2] = model_prices + eps_ask_vector
    return bid_ask_matrix

In [5]:
def check_constraints(R1, R2, S0, sigs1, sigs2, nus1, nus2, bid_ask):

    # 1. Strictly increasing knots and boundary conditions relative to S0
    increasing_nus1 = torch.all(torch.diff(nus1) > 0) and (nus1[-1] < S0)
    increasing_nus2 = torch.all(torch.diff(nus2) > 0) and (nus2[0] > S0)

    # 2. Strictly positive sigmas
    positive_sigs1 = torch.all(sigs1 > 0)
    positive_sigs2 = torch.all(sigs2 > 0)

    # (N x 3) bid_ask matrix
    strikes = bid_ask[:, 0]
    market_bids = bid_ask[:, 1]
    market_asks = bid_ask[:, 2]

    theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

    model_prices = C_K(R1, R2, S0, theta)

    # 3. Bid and ask checks
    bid_check = torch.all(model_prices >= market_bids)
    ask_check = torch.all(model_prices <= market_asks)
    bid_ask_check = bid_check and ask_check

    is_valid = increasing_nus1 and increasing_nus2 and positive_sigs1 and positive_sigs2 and bid_ask_check

    return is_valid.item() if hasattr(is_valid, 'item') else bool(is_valid)

In [13]:
def line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, loss_fn, bid_ask, max_iter, line_search_fn):
    theta_old = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)
    theta_old = theta_old.clone().detach().requires_grad_(True)
    theta_initial = theta_old.clone().detach()

    J_old, _, _, _ = loss_fn(R1, R2, S0, theta_old)
    market_strikes = bid_ask[:, 0]
    market_bids = bid_ask[:, 1]
    market_asks = bid_ask[:, 2]

    line_search = torch.optim.LBFGS([theta_old], max_iter=max_iter, tolerance_grad=1e-15, tolerance_change=1e-15, line_search_fn=line_search_fn)

    iteration = {"count": 0}

    def closure_bid_ask():
      line_search.zero_grad()

      J, _, _, C_K = calculate_J(R1, R2, S0, theta_old)
      scaled_J = 1e3*J

      bid_gap = torch.clamp(C_K - market_bids, min=1e-8)
      ask_gap = torch.clamp(market_asks - C_K, min=1e-8)

      # Log barrier
      bid_barrier = -torch.log(bid_gap)
      ask_barrier = -torch.log(ask_gap)
      modified_J = scaled_J + (torch.sum(bid_barrier) + torch.sum(ask_barrier))

      modified_J.backward()
      i = iteration["count"]
      print(f"Iteration {i:02d} | Modified J: {modified_J.item():<14.6f} | J: {J.item():.12f}")

      with torch.no_grad():
          # Unpack parameter slices for structural layout display
          idx = 0
          nus1_curr = theta_old[idx : idx + R1]; idx += R1
          sigs1_raw_curr = theta_old[idx : idx + R1]; idx += R1
          nus2_curr = theta_old[idx : idx + R2]; idx += R2
          sigs2_raw_curr = theta_old[idx : idx + R2]; idx += R2

          sigs1_curr = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
          sigs2_curr = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

      print(f"  Left Side:")
      for k in range(R1):
          print(f"   nu{k+1} = {nus1_curr[k].item():11.4f} | sigma{k+1} = {sigs1_curr[k].item():11.4f}")
      print(f"  Right Side:")
      for k in range(R2):
          print(f"   nu{k+1} = {nus2_curr[k].item():11.4f} | sigma{k+1} = {sigs2_curr[k].item():11.4f}")
      print("-" * 60)

      iteration["count"] += 1
      return modified_J

    try:
        line_search.step(closure_bid_ask)
    except Exception as e:
        print(f"Error: {e}\n")

    with torch.no_grad():
        J_optimized, _, _, final_prices = calculate_J(R1, R2, S0, theta_old)

        opt_nus1, opt_nus2 = obtain_nus(R1, R2, S0, theta_old)
        opt_sigs1, opt_sigs2 = obtain_sigs(R1, R2, theta_old)

    is_valid = check_constraints(R1, R2, S0, opt_sigs1, opt_sigs2, opt_nus1, opt_nus2, bid_ask)
    print(f"Meeting all constraints: {is_valid}")

    print(f"Current J Value: {J_old.item():.12f}")
    print(f"Optimized J Value: {J_optimized.item():.12f}\n")

    labels = []
    labels.extend([f"nu1_{i}" for i in range(R1)])
    labels.extend([f"sig1_raw_{i}" for i in range(R1)])
    labels.extend([f"nu2_{i}" for i in range(R2)])
    labels.extend([f"sig2_raw_{i}" for i in range(R2)])

    print(f"{'Parameter':<15} | {'Old Value':<15} | {'Optimized Value':<15}")
    print("-" * 53)

    for i in range(len(theta_old)):
        print(f"{labels[i]:<15} | "
              f"{theta_initial[i].item():<15.6f} | "
              f"{theta_old[i].item():<15.6f}")

    print("-" * 53)
    print(f"Total Penalty Reduction: {((J_old - J_optimized)/J_old * 100).item():.12f}%\n")

# Tests

## Test 1

In [14]:
R1, R2 = 4, 3
S0 = 1271.87

sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
sigs2 = torch.tensor([155.0, 180.0, 210.0])
nus1 = torch.linspace(800, 1200, R1)
nus2 = torch.linspace(1400, 2200, R2)
theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

initial_prices = C_K(R1, R2, S0, theta)

eps_bid_vector = torch.clamp(initial_prices * 0.025, min=0.50)
eps_ask_vector = torch.clamp(initial_prices * 0.025, min=0.50)

bid_ask = generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector)

line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, calculate_J, bid_ask, max_iter=30, line_search_fn="strong_wolfe")

Iteration 00 | Modified J: -0.556948      | J: 0.000003114399
  Left Side:
   nu1 =    800.0000 | sigma1 =    149.0000
   nu2 =    933.3333 | sigma2 =    130.0000
   nu3 =   1066.6667 | sigma3 =    250.0000
   nu4 =   1200.0000 | sigma4 =    160.0000
  Right Side:
   nu1 =   1400.0000 | sigma1 =    155.0000
   nu2 =   1800.0000 | sigma2 =    180.0000
   nu3 =   2200.0000 | sigma3 =    210.0000
------------------------------------------------------------
Iteration 01 | Modified J: -0.556948      | J: 0.000003114394
  Left Side:
   nu1 =    800.0000 | sigma1 =    149.0000
   nu2 =    933.3333 | sigma2 =    130.0000
   nu3 =   1066.6667 | sigma3 =    250.0000
   nu4 =   1200.0000 | sigma4 =    160.0000
  Right Side:
   nu1 =   1400.0000 | sigma1 =    155.0000
   nu2 =   1800.0000 | sigma2 =    180.0000
   nu3 =   2200.0000 | sigma3 =    210.0000
------------------------------------------------------------
Iteration 02 | Modified J: -0.556948      | J: 0.000003114346
  Left Side:
   nu1 = 

## Test 2

In [18]:
R1, R2 = 20, 20
S0 = 1400

# sigs1 = torch.randint(100, 300, (20,))
# sigs2 = torch.randint(100, 300, (20,))
sigs1 = torch.repeat_interleave(torch.tensor([100.0, 150.0, 200.0, 250.0], dtype=torch.float64), repeats=5)
sigs2 = torch.repeat_interleave(torch.tensor([100.0, 150.0, 200.0, 250.0], dtype=torch.float64), repeats=5)
nus1 = torch.linspace(700, 1350, R1)
nus2 = torch.linspace(1450, 2300, R2)
theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

initial_prices = C_K(R1, R2, S0, theta)

eps_bid_vector = torch.clamp(initial_prices * 0.025, min=0.50)
eps_ask_vector = torch.clamp(initial_prices * 0.025, min=0.50)

bid_ask = generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector)

line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, calculate_J, bid_ask, max_iter=30, line_search_fn="strong_wolfe")

Iteration 00 | Modified J: 42.700417      | J: 0.000035270112
  Left Side:
   nu1 =    700.0000 | sigma1 =    100.0000
   nu2 =    734.2105 | sigma2 =    100.0000
   nu3 =    768.4211 | sigma3 =    100.0000
   nu4 =    802.6316 | sigma4 =    100.0000
   nu5 =    836.8421 | sigma5 =    100.0000
   nu6 =    871.0526 | sigma6 =    150.0000
   nu7 =    905.2632 | sigma7 =    150.0000
   nu8 =    939.4737 | sigma8 =    150.0000
   nu9 =    973.6842 | sigma9 =    150.0000
   nu10 =   1007.8947 | sigma10 =    150.0000
   nu11 =   1042.1053 | sigma11 =    200.0000
   nu12 =   1076.3158 | sigma12 =    200.0000
   nu13 =   1110.5263 | sigma13 =    200.0000
   nu14 =   1144.7368 | sigma14 =    200.0000
   nu15 =   1178.9474 | sigma15 =    200.0000
   nu16 =   1213.1579 | sigma16 =    250.0000
   nu17 =   1247.3684 | sigma17 =    250.0000
   nu18 =   1281.5789 | sigma18 =    250.0000
   nu19 =   1315.7895 | sigma19 =    250.0000
   nu20 =   1350.0000 | sigma20 =    250.0000
  Right Side:
   nu1 = 

## Test 3

In [19]:
R1, R2 = 70, 50
S0 = 1730

# sigs1 = torch.randint(500, 700, (70,))
# sigs2 = torch.randint(500, 700, (50,))
sigs1 = torch.repeat_interleave(torch.tensor([100.0, 150.0, 200.0, 250.0, 300.0], dtype=torch.float64), repeats=14)
sigs2 = torch.repeat_interleave(torch.tensor([100.0, 150.0, 200.0, 250.0, 300.0], dtype=torch.float64), repeats=10)
nus1 = torch.linspace(600, 1700, R1)
nus2 = torch.linspace(1750, 3120, R2)
theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

initial_prices = C_K(R1, R2, S0, theta)

eps_bid_vector = torch.clamp(initial_prices * 0.025, min=0.50)
eps_ask_vector = torch.clamp(initial_prices * 0.025, min=0.50)

bid_ask = generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector)

line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, calculate_J, bid_ask, max_iter=30, line_search_fn="strong_wolfe")

Iteration 00 | Modified J: 183.636626     | J: 0.000043674200
  Left Side:
   nu1 =    600.0000 | sigma1 =    100.0000
   nu2 =    615.9420 | sigma2 =    100.0000
   nu3 =    631.8841 | sigma3 =    100.0000
   nu4 =    647.8261 | sigma4 =    100.0000
   nu5 =    663.7681 | sigma5 =    100.0000
   nu6 =    679.7101 | sigma6 =    100.0000
   nu7 =    695.6522 | sigma7 =    100.0000
   nu8 =    711.5942 | sigma8 =    100.0000
   nu9 =    727.5362 | sigma9 =    100.0000
   nu10 =    743.4783 | sigma10 =    100.0000
   nu11 =    759.4203 | sigma11 =    100.0000
   nu12 =    775.3623 | sigma12 =    100.0000
   nu13 =    791.3043 | sigma13 =    100.0000
   nu14 =    807.2464 | sigma14 =    100.0000
   nu15 =    823.1884 | sigma15 =    150.0000
   nu16 =    839.1304 | sigma16 =    150.0000
   nu17 =    855.0725 | sigma17 =    150.0000
   nu18 =    871.0145 | sigma18 =    150.0000
   nu19 =    886.9565 | sigma19 =    150.0000
   nu20 =    902.8986 | sigma20 =    150.0000
   nu21 =    918.8406 |

## Test 4

In [22]:
R1, R2 = 4, 3
S0 = 1271.87

sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
sigs2 = torch.tensor([155.0, 180.0, 210.0])
nus1 = torch.linspace(800, 1200, R1)
nus2 = torch.linspace(1400, 2200, R2)
theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

initial_prices = C_K(R1, R2, S0, theta)

eps_bid_vector = torch.tensor([0.5, 0.1, 0.1, 0.2, 0.3, 0.4, 0.35, 0.2])
eps_ask_vector = torch.tensor([0.6, 0.3, 0.2, 0.1, 0.2, 0.1, 0.3, 0.24])

bid_ask = generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector)

line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, calculate_J, bid_ask, max_iter=30, line_search_fn="strong_wolfe")

Iteration 00 | Modified J: 23.860327      | J: 0.000003114399
  Left Side:
   nu1 =    800.0000 | sigma1 =    149.0000
   nu2 =    933.3333 | sigma2 =    130.0000
   nu3 =   1066.6667 | sigma3 =    250.0000
   nu4 =   1200.0000 | sigma4 =    160.0000
  Right Side:
   nu1 =   1400.0000 | sigma1 =    155.0000
   nu2 =   1800.0000 | sigma2 =    180.0000
   nu3 =   2200.0000 | sigma3 =    210.0000
------------------------------------------------------------
Iteration 01 | Modified J: 23.912598      | J: 0.000003127859
  Left Side:
   nu1 =    799.9922 | sigma1 =    149.0044
   nu2 =    933.3401 | sigma2 =    130.0748
   nu3 =   1066.8217 | sigma3 =    250.0072
   nu4 =   1200.1228 | sigma4 =    159.8259
  Right Side:
   nu1 =   1400.0408 | sigma1 =    154.6599
   nu2 =   1799.9967 | sigma2 =    179.9376
   nu3 =   2200.0001 | sigma3 =    210.0005
------------------------------------------------------------
Iteration 02 | Modified J: 23.596241      | J: 0.000003120751
  Left Side:
   nu1 = 

## Test 5

In [23]:
R1, R2 = 4, 3
S0 = 1271.87

sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
sigs2 = torch.tensor([155.0, 180.0, 210.0])
nus1 = torch.linspace(800, 1200, R1)
nus2 = torch.linspace(1400, 2200, R2)
theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)

initial_prices = C_K(R1, R2, S0, theta)

eps_bid_vector = torch.tensor([1, 2, 4, 5, 3, 2, 1, 3])
eps_ask_vector = torch.tensor([1, 2, 4, 5, 3, 2, 1, 3])

bid_ask = generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector)

line_search_test(R1, R2, S0, sigs1, sigs2, nus1, nus2, calculate_J, bid_ask, max_iter=30, line_search_fn="strong_wolfe")

Iteration 00 | Modified J: -13.155388     | J: 0.000003114399
  Left Side:
   nu1 =    800.0000 | sigma1 =    149.0000
   nu2 =    933.3333 | sigma2 =    130.0000
   nu3 =   1066.6667 | sigma3 =    250.0000
   nu4 =   1200.0000 | sigma4 =    160.0000
  Right Side:
   nu1 =   1400.0000 | sigma1 =    155.0000
   nu2 =   1800.0000 | sigma2 =    180.0000
   nu3 =   2200.0000 | sigma3 =    210.0000
------------------------------------------------------------
Iteration 01 | Modified J: -13.155388     | J: 0.000003114394
  Left Side:
   nu1 =    800.0000 | sigma1 =    149.0000
   nu2 =    933.3333 | sigma2 =    130.0000
   nu3 =   1066.6667 | sigma3 =    250.0000
   nu4 =   1200.0000 | sigma4 =    160.0000
  Right Side:
   nu1 =   1400.0000 | sigma1 =    155.0000
   nu2 =   1800.0000 | sigma2 =    180.0000
   nu3 =   2200.0000 | sigma3 =    210.0000
------------------------------------------------------------
Iteration 02 | Modified J: -13.155388     | J: 0.000003114346
  Left Side:
   nu1 = 

# Bisection Line Search

In [ ]:
def bisection_line_search(theta, gradient, loss_fn, R1, R2, S0, alpha_max=10, tol=1e-5, max_iter=20):

    alpha_left = 0
    alpha_right = alpha_max

    best_theta = theta.clone().detach()
    best_loss = float('inf')
    best_alpha = 0

    for iteration in range(max_iter):
        alpha_mid = (alpha_left + alpha_right)/2

        new_theta = theta - alpha_mid * gradient
        new_param = new_theta.clone().detach().requires_grad_(True)

        J_candidate, _, _ = loss_fn(R1, R2, S0, new_param)
        scaled_J = J_candidate * 1e7
        scaled_J.backward()

        # p = grad(theta_mid) (dot product) (initial_gradient)
        curr_grad = new_param.grad.clone().detach()
        p = torch.sum(- curr_grad * gradient).item()

        if J_candidate.item() < best_loss:
            best_loss = J_candidate.item()
            best_theta = new_theta.clone().detach()
            best_alpha = alpha_mid

        if (alpha_right - alpha_left) < tol:
            break

        if p > 0:
            # > min, move right bound inward
            alpha_right = alpha_mid
        else:
            alpha_left = alpha_mid

    return best_theta, best_alpha, best_loss

In [ ]:
def bisection_line_search_test(R1, R2, S0, theta, iter, max_iter, alpha_max):
  for i in range(iter):
    J, _, _, _ = calculate_J(R1, R2, S0, theta)
    scaled_J = J * 1e7
    scaled_J.backward()

    init_gradient = theta.grad.clone().detach()
    theta.grad.zero_()

    theta_new, alpha, curr_loss = bisection_line_search(theta=theta, gradient=init_gradient, loss_fn=calculate_J, R1=R1, R2=R2, S0=S0,
                                                        alpha_max=alpha_max, tol=1e-6, max_iter=max_iter)

    print(f"Iteration {i:02d} | Alpha: {alpha:<9.6f} | Current J: {curr_loss:.12f}")

    with torch.no_grad():
        theta.copy_(theta_new)
        idx = 0
        nus1 = theta[idx : idx + R1]; idx += R1
        sigs1_raw_curr = theta[idx : idx + R1]; idx += R1
        nus2 = theta[idx : idx + R2]; idx += R2
        sigs2_raw_curr = theta[idx : idx + R2]; idx += R2

        sigs1 = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
        sigs2 = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

    print(f"  Left Side:")
    for i in range(R1):
        print(f"   nu{i+1} = {nus1[i].item():11.4f} | sigma{i+1} = {sigs1[i].item():11.4f}")

    print(f"  Right Side:")
    for i in range(R2):
        print(f"   nu{i+1} = {nus2[i].item():11.4f} | sigma{i+1} = {sigs2[i].item():11.4f}")

    print("-" * 60)

In [ ]:
def bisection_line_search_test(R1, R2, S0, theta, iter, max_iter, alpha_max):
  for i in range(iter):
    J, _, _, _ = calculate_J(R1, R2, S0, theta)
    scaled_J = J * 1e7
    scaled_J.backward()

    init_gradient = theta.grad.clone().detach()
    theta.grad.zero_()

    theta_new, alpha, curr_loss = bisection_line_search(theta=theta, gradient=init_gradient, loss_fn=calculate_J, R1=R1, R2=R2, S0=S0,
                                                        alpha_max=alpha_max, tol=1e-6, max_iter=max_iter)

    print(f"Iteration {i:02d} | Alpha: {alpha:<9.6f} | Current J: {curr_loss:.12f}")

    with torch.no_grad():
        theta.copy_(theta_new)
        idx = 0
        nus1 = theta[idx : idx + R1]; idx += R1
        sigs1_raw_curr = theta[idx : idx + R1]; idx += R1
        nus2 = theta[idx : idx + R2]; idx += R2
        sigs2_raw_curr = theta[idx : idx + R2]; idx += R2

        sigs1 = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
        sigs2 = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

    print(f"  Left Side:")
    for i in range(R1):
        print(f"   nu{i+1} = {nus1[i].item():11.4f} | sigma{i+1} = {sigs1[i].item():11.4f}")

    print(f"  Right Side:")
    for i in range(R2):
        print(f"   nu{i+1} = {nus2[i].item():11.4f} | sigma{i+1} = {sigs2[i].item():11.4f}")

    print("-" * 60)